### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="credit_approval",
    dataset_year="1987",
    domain_str="finance",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5FS30",
    download_description="""
We get the credit screening data from UCI.

wget https://archive.ics.uci.edu/static/public/27/credit+approval.zip && unzip credit+approval.zip  crx.data && rm credit+approval.zip  && mkdir -p local-data-warehouse/credit_approval && mv crx.data  local-data-warehouse/credit_approval/
""",
    # References
    academic_reference_bibtex="""@article{quinlan1987simplifying,
  title={Simplifying decision trees},
  author={Quinlan, J. Ross},
  journal={International journal of man-machine studies},
  volume={27},
  number={3},
  pages={221--234},
  year={1987},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="quinlan1987simplifying",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We get the data from UCI.

- We encode A14 as numeric, following the description of it being a continuous feature.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="A16",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="A16",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

columns = ["A1","A2","A3","A4","A5","A6","A7","A8","A9","A10","A11","A12","A13","A14","A15","A16"]
df = pd.read_csv(dataset_mold.path / "crx.data", header=None, names=columns, na_values="?")

df["A14"] = pd.to_numeric(df["A14"], errors="coerce")

as_cat_types = ["A1", "A4", "A5", "A6", "A7", "A9", "A10", "A12", "A13", "A16"]
df[as_cat_types] = df[as_cat_types].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 690
Columns: 16
Use sampling: False (sample size: 690)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['A2', 'A15', 'A3', 'A14', 'A8', 'A11', 'A6', 'A7', 'A5', 'A4']
Rows remaining as candidates after top-10 filter: 2 (of 690)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15,A16
0,a,NaN,1.5,u,g,ff,ff,0.0,f,t,2,t,g,200.0,105,-
1,a,46.00,4.0,u,g,j,j,0.0,t,f,0,f,g,100.0,960,+
2,b,20.00,0.0,u,g,d,v,0.5,f,f,0,f,g,144.0,0,-
3,b,47.33,6.5,u,g,c,v,1.0,f,f,0,t,g,0.0,228,-
4,b,19.17,0.0,y,p,m,bb,0.0,f,f,0,t,s,500.0,1,+


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,A1,category,12.0,1.74,2.0,"b, a"
1,A6,category,9.0,1.30,14.0,"c, q, w, i, aa, ff, k, cc, x, m"
2,A7,category,9.0,1.30,9.0,"v, h, bb, ff, j, z, dd, n, o"
3,A4,category,6.0,0.87,3.0,"u, y, l"
4,A5,category,6.0,0.87,3.0,"g, p, gg"
5,A9,category,0.0,0.00,2.0,"t, f"
6,A10,category,0.0,0.00,2.0,"f, t"
7,A12,category,0.0,0.00,2.0,"f, t"
8,A13,category,0.0,0.00,3.0,"g, s, p"
9,A16,category,0.0,0.00,2.0,"-, +"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
A2,678.0,31.568171,11.957862,13.75,80.25
A3,690.0,4.758725,4.978163,0.00,28.00
A8,690.0,2.223406,3.346513,0.00,28.50
A11,690.0,2.400000,4.862940,0.00,67.00
A14,677.0,184.014771,173.806768,0.00,2000.00
A15,690.0,1017.385507,5210.102598,0.00,100000.00


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column rank                    
A1     1        b    468  67.83
       2        a    210  30.43
       3     <NA>     12   1.74
A10    1        f    395  57.25
       2        t    295  42.75
A12    1        f    374  54.20
       2        t    316  45.80
A13    1        g    625  90.58
       2        s     57   8.26
       3        p      8   1.16
A16    1        -    383  55.51
       2        +    307  44.49
A4     1        u    519  75.22
       2        y    163  23.62
       3     <NA>      6   0.87
       4        l      2   0.29
A5     1        g    519  75.22
       2        p    163  23.62
       3     <NA>      6   0.87
       4       gg      2   0.29
A6     1        c    137  19.86
       2        q     78  11.30
       3        w     64   9.28
       4        i     59   8.55
       5       aa     54   7.83
A7     1        v    399  57.83
       2        h    138  20.00
       3       bb     59   8.55
       4       ff     57   8.26
       5     <NA>      9   1.30
A9     1        t    361  52.32
       2        f    329  47.68

In [8]:
# Target Distribution
target_df

,count,pct
A16,,
-,383,55.51
+,307,44.49


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_iid_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_iid_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019cb05b-bb5e-736e-8b40-7f5528c6fe6f
b0ed3554bd1efdbca20246a2018397342041ca3a6cfea489ff9df2e3518fc93f
